In [1]:
from pathlib import Path
import csv
import json
import shutil
from collections import Counter

LAST_NAME = "Nicolas"
STUDENT_ID = "TUPM-26-7931"

SEED_DIGIT = int(STUDENT_ID[-1])
ID_SUM = sum(int(d) for d in STUDENT_ID if d.isdigit())
NAME_LENGTH = len(LAST_NAME)

print("Student:", LAST_NAME)
print("Student ID:", STUDENT_ID)
print("Seed Digit:", SEED_DIGIT)
print("ID Sum:", ID_SUM)
print("Name Length:", NAME_LENGTH)

Student: Nicolas
Student ID: TUPM-26-7931
Seed Digit: 1
ID Sum: 28
Name Length: 7


In [2]:
WORK_DIR = Path(".")
OUTPUT_DIR = WORK_DIR / "output"
BACKUP_DIR = WORK_DIR / "backup"

OUTPUT_DIR.mkdir(exist_ok=True)
BACKUP_DIR.mkdir(exist_ok=True)

dataset_path = WORK_DIR / "Global_Cybersecurity_Threats_2015-2024.csv"

print("Workspace:", WORK_DIR.resolve())
print("Dataset exists:", dataset_path.exists())
print("Output folder exists:", OUTPUT_DIR.exists())
print("Backup folder exists:", BACKUP_DIR.exists())

Workspace: C:\Users\Chichi\Lab6_File_Handling_Nicolas\Lab6_File_Handling_Additional_Exercises\Exercise_3_Cybersecurity_Incident_Report
Dataset exists: True
Output folder exists: True
Backup folder exists: True


In [3]:
TARGET_YEAR = 2015 + SEED_DIGIT

print("Seed Digit:", SEED_DIGIT)
print("Target Year:", TARGET_YEAR)

Seed Digit: 1
Target Year: 2016


In [4]:
with open(dataset_path, "r", encoding="utf-8-sig", newline="") as file:
    reader = csv.DictReader(file)

    print("Dataset Columns:")
    print(reader.fieldnames)

    print("\nSample Records:")

    for i, record in enumerate(reader):
        if i == 3:
            break
        print(record)

Dataset Columns:
['Country', 'Year', 'Attack Type', 'Target Industry', 'Financial Loss (in Million $)', 'Number of Affected Users', 'Attack Source', 'Security Vulnerability Type', 'Defense Mechanism Used', 'Incident Resolution Time (in Hours)']

Sample Records:
{'Country': 'China', 'Year': '2019', 'Attack Type': 'Phishing', 'Target Industry': 'Education', 'Financial Loss (in Million $)': '80.53', 'Number of Affected Users': '773169', 'Attack Source': 'Hacker Group', 'Security Vulnerability Type': 'Unpatched Software', 'Defense Mechanism Used': 'VPN', 'Incident Resolution Time (in Hours)': '63'}
{'Country': 'China', 'Year': '2019', 'Attack Type': 'Ransomware', 'Target Industry': 'Retail', 'Financial Loss (in Million $)': '62.19', 'Number of Affected Users': '295961', 'Attack Source': 'Hacker Group', 'Security Vulnerability Type': 'Unpatched Software', 'Defense Mechanism Used': 'Firewall', 'Incident Resolution Time (in Hours)': '71'}
{'Country': 'India', 'Year': '2017', 'Attack Type': 'M

In [5]:
selected_incidents = []
valid_loss_records = []

total_records = 0
invalid_year_records = 0
invalid_financial_values = 0

with open(dataset_path, "r", encoding="utf-8-sig", newline="") as file:
    reader = csv.DictReader(file)

    required_columns = [
        "Year",
        "Attack Type",
        "Financial Loss (in Million $)"
    ]

    if not all(col in reader.fieldnames for col in required_columns):
        raise ValueError(
            f"Missing required columns: {reader.fieldnames}"
        )

    for record in reader:
        total_records += 1

        try:
            year = int(record["Year"].strip())
        except (ValueError, AttributeError):
            invalid_year_records += 1
            continue

        if year != TARGET_YEAR:
            continue

        selected_incidents.append(record)

        try:
            loss = float(
                record["Financial Loss (in Million $)"].strip()
            )

            if not loss >= 0 or loss == float("inf") or loss != loss:
                raise ValueError("Invalid financial loss")

            valid_loss_records.append(
                (total_records - 1, record, loss)
            )

        except (ValueError, TypeError, AttributeError):
            invalid_financial_values += 1

print("Total records in dataset:", total_records)
print("Target year:", TARGET_YEAR)
print("Incidents in target year:", len(selected_incidents))
print("Invalid year records:", invalid_year_records)
print("Invalid financial values:", invalid_financial_values)
print("Valid financial records:", len(valid_loss_records))

Total records in dataset: 3000
Target year: 2016
Incidents in target year: 285
Invalid year records: 0
Invalid financial values: 0
Valid financial records: 285


In [6]:
attack_counts = Counter()

for record in selected_incidents:
    attack_type = record["Attack Type"].strip()

    if not attack_type:
        attack_type = "Unknown"

    attack_counts[attack_type] += 1

print("INCIDENT COUNTS PER ATTACK TYPE")

for attack_type, count in sorted(attack_counts.items()):
    print(f"{attack_type}: {count} incidents")

print("Total incidents counted:", sum(attack_counts.values()))

INCIDENT COUNTS PER ATTACK TYPE
DDoS: 53 incidents
Malware: 41 incidents
Man-in-the-Middle: 47 incidents
Phishing: 55 incidents
Ransomware: 42 incidents
SQL Injection: 47 incidents
Total incidents counted: 285


In [7]:
total_financial_loss = sum(
    loss for _, record, loss in valid_loss_records
)

valid_loss_count = len(valid_loss_records)

average_financial_loss = (
    total_financial_loss / valid_loss_count
    if valid_loss_count > 0 else 0
)

print(
    "Total Financial Loss:",
    round(total_financial_loss, 2),
    "million USD"
)

print(
    "Average Financial Loss:",
    round(average_financial_loss, 2),
    "million USD"
)

Total Financial Loss: 13947.26 million USD
Average Financial Loss: 48.94 million USD


In [8]:
# Sort by descending financial loss.
# Python's stable sorting preserves original order when losses are tied.

ranked_incidents = sorted(
    valid_loss_records,
    key=lambda item: -item[2]
)

top_five = ranked_incidents[:5]

print("FIVE COSTLIEST INCIDENTS")

for rank, (original_index, record, loss) in enumerate(top_five, 1):
    print(f"\nRank {rank}")
    print("Country:", record["Country"])
    print("Attack Type:", record["Attack Type"])
    print("Target Industry:", record["Target Industry"])
    print("Financial Loss:", loss, "million USD")

FIVE COSTLIEST INCIDENTS

Rank 1
Country: Germany
Attack Type: Phishing
Target Industry: Telecommunications
Financial Loss: 99.98 million USD

Rank 2
Country: USA
Attack Type: Ransomware
Target Industry: Telecommunications
Financial Loss: 99.88 million USD

Rank 3
Country: Brazil
Attack Type: Malware
Target Industry: IT
Financial Loss: 99.29 million USD

Rank 4
Country: USA
Attack Type: Ransomware
Target Industry: Healthcare
Financial Loss: 99.19 million USD

Rank 5
Country: Australia
Attack Type: Phishing
Target Industry: Education
Financial Loss: 98.72 million USD


In [9]:
filtered_csv_path = (
    OUTPUT_DIR /
    f"{LAST_NAME}_{STUDENT_ID}_incidents_2016.csv"
)

if selected_incidents:
    with open(filtered_csv_path, "w", encoding="utf-8", newline="") as file:
        writer = csv.DictWriter(
            file,
            fieldnames=selected_incidents[0].keys()
        )

        writer.writeheader()
        writer.writerows(selected_incidents)

else:
    with open(dataset_path, "r", encoding="utf-8-sig", newline="") as source:
        fieldnames = csv.DictReader(source).fieldnames

    with open(filtered_csv_path, "w", encoding="utf-8", newline="") as file:
        writer = csv.DictWriter(file, fieldnames=fieldnames)
        writer.writeheader()

print("Filtered CSV created:", filtered_csv_path)
print("Exported incidents:", len(selected_incidents))

Filtered CSV created: output\Nicolas_TUPM-26-7931_incidents_2016.csv
Exported incidents: 285


In [10]:
json_path = (
    OUTPUT_DIR /
    f"{LAST_NAME}_{STUDENT_ID}_top_five_incidents.json"
)

top_five_data = []

for rank, (original_index, record, loss) in enumerate(top_five, 1):
    incident = dict(record)
    incident["Rank"] = rank
    incident["Financial Loss (in Million $)"] = loss

    top_five_data.append(incident)

with open(json_path, "w", encoding="utf-8") as file:
    json.dump(top_five_data, file, indent=4)

print("JSON file created:", json_path)
print("Costliest incidents exported:", len(top_five_data))

JSON file created: output\Nicolas_TUPM-26-7931_top_five_incidents.json
Costliest incidents exported: 5


In [11]:
backup_path = BACKUP_DIR / json_path.name

shutil.copy2(json_path, backup_path)

with open(json_path, "rb") as original:
    original_content = original.read()

with open(backup_path, "rb") as backup:
    backup_content = backup.read()

backup_verified = (
    backup_path.exists()
    and original_content == backup_content
)

print("Original JSON:", json_path)
print("Backup JSON:", backup_path)
print("Backup exists:", backup_path.exists())
print("Backup size:", backup_path.stat().st_size, "bytes")
print("Backup verification:", backup_verified)

Original JSON: output\Nicolas_TUPM-26-7931_top_five_incidents.json
Backup JSON: backup\Nicolas_TUPM-26-7931_top_five_incidents.json
Backup exists: True
Backup size: 2330 bytes
Backup verification: True


In [12]:
report_path = (
    OUTPUT_DIR /
    f"{LAST_NAME}_{STUDENT_ID}_incident_report.txt"
)

report_lines = [
    "CYBERSECURITY INCIDENT REPORT",
    "----------------------------------------",
    f"Student: {LAST_NAME}",
    f"Student ID: {STUDENT_ID}",
    f"Seed Digit: {SEED_DIGIT}",
    f"Target Year: {TARGET_YEAR}",
    "",
    "DATASET INFORMATION",
    f"Total Records: {total_records}",
    f"Incidents in Target Year: {len(selected_incidents)}",
    f"Invalid Year Records: {invalid_year_records}",
    f"Invalid Financial Values: {invalid_financial_values}",
    "",
    "INCIDENT COUNTS PER ATTACK TYPE"
]

for attack_type, count in sorted(attack_counts.items()):
    report_lines.append(f"{attack_type}: {count}")

report_lines.extend([
    "",
    "FINANCIAL LOSS SUMMARY",
    f"Valid Financial Records: {valid_loss_count}",
    f"Total Financial Loss: {total_financial_loss:.2f} million USD",
    f"Average Financial Loss: {average_financial_loss:.2f} million USD",
    "",
    "FIVE COSTLIEST INCIDENTS"
])

for rank, (original_index, record, loss) in enumerate(top_five, 1):
    report_lines.extend([
        f"Rank {rank}:",
        f"Country: {record['Country']}",
        f"Attack Type: {record['Attack Type']}",
        f"Target Industry: {record['Target Industry']}",
        f"Financial Loss: {loss:.2f} million USD",
        ""
    ])

report_lines.extend([
    "OUTPUT FILE INFORMATION",
    f"Filtered CSV: {filtered_csv_path}",
    f"JSON Summary: {json_path}",
    f"Backup JSON: {backup_path}",
    f"Backup Verified: {backup_verified}"
])

with open(report_path, "w", encoding="utf-8") as file:
    file.write("\n".join(report_lines))

print("Report created:", report_path)

Report created: output\Nicolas_TUPM-26-7931_incident_report.txt


In [13]:
with open(report_path, "r", encoding="utf-8") as file:
    print(file.read())

CYBERSECURITY INCIDENT REPORT
----------------------------------------
Student: Nicolas
Student ID: TUPM-26-7931
Seed Digit: 1
Target Year: 2016

DATASET INFORMATION
Total Records: 3000
Incidents in Target Year: 285
Invalid Year Records: 0
Invalid Financial Values: 0

INCIDENT COUNTS PER ATTACK TYPE
DDoS: 53
Malware: 41
Man-in-the-Middle: 47
Phishing: 55
Ransomware: 42
SQL Injection: 47

FINANCIAL LOSS SUMMARY
Valid Financial Records: 285
Total Financial Loss: 13947.26 million USD
Average Financial Loss: 48.94 million USD

FIVE COSTLIEST INCIDENTS
Rank 1:
Country: Germany
Attack Type: Phishing
Target Industry: Telecommunications
Financial Loss: 99.98 million USD

Rank 2:
Country: USA
Attack Type: Ransomware
Target Industry: Telecommunications
Financial Loss: 99.88 million USD

Rank 3:
Country: Brazil
Attack Type: Malware
Target Industry: IT
Financial Loss: 99.29 million USD

Rank 4:
Country: USA
Attack Type: Ransomware
Target Industry: Healthcare
Financial Loss: 99.19 million USD

Rank

In [14]:
with open(filtered_csv_path, "r", encoding="utf-8", newline="") as file:
    exported_incidents = sum(
        1 for row in csv.DictReader(file)
    )

with open(json_path, "r", encoding="utf-8") as file:
    loaded_top_five = json.load(file)

print("FINAL EXERCISE 3 VERIFICATION")
print("--------------------------------")

print("Target year:", TARGET_YEAR)
print("Total dataset records:", total_records)
print("Incidents in target year:", len(selected_incidents))
print("Attack types counted:", sum(attack_counts.values()))

print("Filtered CSV records:", exported_incidents)
print("CSV count verification:",
      exported_incidents == len(selected_incidents))

print("JSON records:", len(loaded_top_five))
print("JSON verification:",
      len(loaded_top_five) == len(top_five))

print("Backup verification:", backup_verified)
print("Report exists:", report_path.exists())

print("Overall verification:",
      exported_incidents == len(selected_incidents)
      and len(loaded_top_five) == len(top_five)
      and backup_verified
      and report_path.exists())

FINAL EXERCISE 3 VERIFICATION
--------------------------------
Target year: 2016
Total dataset records: 3000
Incidents in target year: 285
Attack types counted: 285
Filtered CSV records: 285
CSV count verification: True
JSON records: 5
JSON verification: True
Backup verification: True
Report exists: True
Overall verification: True
